# GitHub API Client

Handles authentication (authenticating with your GitHub account gives you much higher rate limits). It also handles rate limits gracefully.

The way the client authenticates with the first token it finds: one passed to `GitHubClient`, then `GITHUB_TOKEN` set as an environment variable, then `gh auth token` from the GitHub CLI. Without any of those it still works with lower rate limits.

By default responses are cached under `.cache/github/` for a day, so re-running a cell does not spend more of the quota.

In [1]:
import asyncio
import csv
from pathlib import Path
import random
import time

import gh_signals.github_client_v2 as m
from gh_signals.github_client_v2 import GitHubClient

## Load Repo List

`data/good_repos.csv` holds repositories we treat as the positive class and `data/uncertain.csv` as the negative class.
Let's sample a few of the good repos and get their number of stars and forks.

In [2]:
csv_path = Path("..", "data", "good_repos.csv")
with csv_path.open() as file:
    good_repos = [row["repo"] for row in csv.DictReader(file)]

chosen = random.sample(good_repos, 3)
chosen

['microsoftgraph/msgraph-sdk-dotnet', 'torvalds/linux', 'microsoft/playwright']

## Fetch Data Using the GitHub Client

In [3]:
async with GitHubClient() as client:
    # Fetch the repository information for each chosen repository concurrently
    results = await asyncio.gather(*(client.get_repo(name) for name in chosen))

for result in results:
    print(f"{result.full_name}: {result.stargazers_count} stars, {result.forks_count} forks")

microsoftgraph/msgraph-sdk-dotnet: 790 stars, 264 forks
torvalds/linux: 250046 stars, 65589 forks
microsoft/playwright: 96639 stars, 6500 forks


## The following are tests I have written to check that my script has acomplished everything on the task list

## Task 1: Personal access token integration

Expectation: Find a token automatically and use it, so you get 5,000 requests an hour instead of 60.

## Test 1a: Is the token found and used?

Pass if there's no "no GitHub token" warning and the number is near 5000. cache_dir=None forces a real request, so you're testing the token and not a saved file.

In [ ]:
async with GitHubClient(cache_dir=None) as client:
    await client.get_repo_raw("numpy/numpy")
    print("requests remaining:", client._remaining)

## Test 1b: Does a bad token get rejected properly?

Pass if we get GitHubError: GitHub returned 401 ... right away. It should not retry, because a wrong token will never start working.

In [ ]:
async with GitHubClient(token="not_a_real_token", cache_dir=None) as client:
    await client.get_repo_raw("numpy/numpy")

## Test 1c: Does the token you pass in take priority?
The bad-token test above already shows this. A passed-in token beat the working gh login, and that's why it failed.

## Task 2: Retry and backoff logic

Expectation: Survive temporary problems by waiting and trying again, and stop right away on problems that retrying can't fix.

Test 2a: A missing repo fails fast (no retry). Pass if we get GitHubError ... 404 immediately with no "retrying" messages.

In [ ]:
async with GitHubClient(cache_dir=None) as client:
    await client.get_repo_raw("fake/doesnotexist123")

## Test 2b: Network failure retries with growing delays

Pass if it takes about 3 seconds (a 1 second wait, then a 2 second wait), then ends with a connection error.

In [ ]:
original_url = m.API_URL
m.API_URL = "http://localhost:1"   # nothing is running here, so every request fails

start = time.perf_counter()
try:
    async with GitHubClient(max_retries=2, cache_dir=None) as client:
        await client.get_repo_raw("numpy/numpy")
except Exception as error:
    print("gave up with:", type(error).__name__)
print(f"took {time.perf_counter() - start:.1f}s")

m.API_URL = original_url           # put the real address back

## Test 2c: it pauses when the rate limit is nearly used up

Pass if the second call prints Rate limit almost used up, sleeping ...s. Then click the stop button, or the cell will wait until the hourly reset.

In [9]:
async with GitHubClient(min_remaining=5000, cache_dir=None) as client:
    await client.get_repo_raw("numpy/numpy")
    await client.get_repo_raw("pandas-dev/pandas")

Rate limit almost used up, sleeping 1529s until it resets


CancelledError: 

## Task 3: Local caching

What it should do: Save every response to disk, so a crashed or restarted batch doesn't spend the API limit again.

# Test 3a: A fetch creates a file, and a second fetch is instant

Pass if the second time is far faster and remaining doesn't drop between the two lines. Also check that .cache/github/repos_numpy_numpy.json exists in the left panel.

In [10]:
async with GitHubClient() as client:
    start = time.perf_counter()
    await client.get_repo_raw("numpy/numpy")
    print(f"first: {time.perf_counter() - start:.2f}s, remaining: {client._remaining}")

    start = time.perf_counter()
    await client.get_repo_raw("numpy/numpy")
    print(f"second: {time.perf_counter() - start:.4f}s, remaining: {client._remaining}")

first: 0.00s, remaining: 11
second: 0.0003s, remaining: 11


## Test 3b: An expired cache gets refetched

Pass if remaining goes down by one compared to before. A cache lifetime of 0 seconds means everything counts as stale.

In [ ]:
async with GitHubClient(cache_ttl_seconds=0) as client:
    await client.get_repo_raw("numpy/numpy")
    print("remaining:", client._remaining)

## Test 3c: A stopped batch resumes without repeating work

Start it, then stop the cell partway with the stop button. Count what got saved. Afterwards check if it finishes much faster than the first attempt, because the saved repos load instantly. 

In [7]:
async with GitHubClient() as client:
    await asyncio.gather(*(client.get_repo_raw(r) for r in good_repos[:50]))

In [8]:
print(len(list(m.DEFAULT_CACHE_DIR.glob("*.json"))), "repos saved")

60 repos saved


## Test 3d: A corrupt file doesn't crash anything. 

What To Do: Open .cache/github/repos_numpy_numpy.json and delete half the text, then save it. Run Test 3a again. Pass if it quietly refetches and prints no error.